# Decisión económica para retener clientes

En esta parte usé las probabilidades que guardé en el notebook 01. Primero busqué el umbral que daba el mayor valor neto esperado en validación. Después probé ese umbral en los datos de prueba.

El dataset no tiene resultados de campañas. Por eso la cuenta depende de algunos valores de ejemplo que dejé separados para poder cambiarlos.

## 1. Cargué las predicciones

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

# Subí el archivo que guardé al final del notebook 01.
archivos = files.upload()
nombre_archivo = next(iter(archivos))
predicciones = pd.read_csv(nombre_archivo)

print("Filas y columnas:", predicciones.shape)
display(predicciones.head())

## 2. Revisé que estuvieran las columnas

In [ ]:
columnas_necesarias = [
    "conjunto",
    "customer_id",
    "churn_real",
    "cargo_mensual",
    "probabilidad_churn",
]
faltantes = [columna for columna in columnas_necesarias if columna not in predicciones.columns]

if faltantes:
    raise ValueError(f"Faltan columnas que necesito: {faltantes}")

validacion = predicciones[predicciones["conjunto"] == "validacion"].copy()
prueba = predicciones[predicciones["conjunto"] == "prueba"].copy()

print("Clientes de validación:", len(validacion))
print("Clientes de prueba:", len(prueba))

## 3. Definí los datos económicos de ejemplo

La consigna pide calcular un valor esperado, pero no da el costo de la campaña, el margen ni su tasa de éxito. El CSV tampoco contiene esos datos. Para que el cálculo se pueda ejecutar, dejé estos valores de ejemplo. No son datos reales de la empresa y se pueden cambiar.

In [ ]:
COSTO_CAMPANIA = 20.0
MESES_ESPERADOS = 6
MARGEN_ESTIMADO = 0.40
EXITO_CAMPANIA = 0.30

# Calculé cuánto margen futuro habría si se conservara a cada cliente.
validacion["margen_futuro_estimado"] = (
    validacion["cargo_mensual"] * MESES_ESPERADOS * MARGEN_ESTIMADO
)
prueba["margen_futuro_estimado"] = (
    prueba["cargo_mensual"] * MESES_ESPERADOS * MARGEN_ESTIMADO
)

# Calculé el beneficio esperado de contactar a cada cliente.
validacion["valor_si_contacto"] = (
    validacion["probabilidad_churn"]
    * EXITO_CAMPANIA
    * validacion["margen_futuro_estimado"]
    - COSTO_CAMPANIA
)
prueba["valor_si_contacto"] = (
    prueba["probabilidad_churn"]
    * EXITO_CAMPANIA
    * prueba["margen_futuro_estimado"]
    - COSTO_CAMPANIA
)

print("Supuestos que usé:")
print("Costo por campaña:", COSTO_CAMPANIA)
print("Meses de margen estimados:", MESES_ESPERADOS)
print("Margen estimado:", MARGEN_ESTIMADO)
print("Probabilidad de éxito de campaña:", EXITO_CAMPANIA)

## 4. Busqué el umbral en validación

Un umbral es el corte de probabilidad desde el que decidí contactar. Por ejemplo, si el umbral es 0,40, contactaría a quienes tienen una probabilidad de churn de 0,40 o más.

Probé valores desde 0 hasta 1. También agregué 1,01 para representar la opción de no contactar a nadie.

In [ ]:
umbrales = list(np.round(np.arange(0, 1.01, 0.01), 2))
umbrales.append(1.01)

resultados_umbral = []

for umbral in umbrales:
    contactar = validacion["probabilidad_churn"] >= umbral

    # Sumé el valor esperado solo para las personas que habría contactado.
    valor_neto = validacion.loc[contactar, "valor_si_contacto"].sum()
    cantidad_contactada = int(contactar.sum())

    resultados_umbral.append({
        "umbral": umbral,
        "clientes_contactados": cantidad_contactada,
        "valor_neto_esperado": valor_neto,
    })

tabla_umbrales = pd.DataFrame(resultados_umbral)
mejor_fila = tabla_umbrales.loc[tabla_umbrales["valor_neto_esperado"].idxmax()]
UMBRAL_ELEGIDO = float(mejor_fila["umbral"])

display(tabla_umbrales.sort_values("valor_neto_esperado", ascending=False).head(10))
print("Umbral elegido:", UMBRAL_ELEGIDO)
print("Clientes que contactaría en validación:", int(mejor_fila["clientes_contactados"]))
print("Valor neto esperado en validación:", round(mejor_fila["valor_neto_esperado"], 2))

In [ ]:
# Mostré cómo cambia el valor esperado según el umbral.
plt.figure(figsize=(8, 4))
plt.plot(
    tabla_umbrales["umbral"],
    tabla_umbrales["valor_neto_esperado"],
    color="teal",
)
plt.axvline(UMBRAL_ELEGIDO, color="tomato", linestyle="--", label=f"Umbral: {UMBRAL_ELEGIDO:.2f}")
plt.xlabel("Umbral de probabilidad")
plt.ylabel("Valor neto esperado")
plt.title("Valor esperado por umbral (validación)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 5. Probé el umbral en los datos de prueba

Apliqué el umbral elegido en validación a los datos de prueba. La cantidad y el valor pueden cambiar si cambian los datos o los supuestos.

In [ ]:
prueba["recomendar_contacto"] = prueba["probabilidad_churn"] >= UMBRAL_ELEGIDO

# Sumé el beneficio esperado y resté el costo de las campañas recomendadas.
prueba["beneficio_esperado"] = np.where(
    prueba["recomendar_contacto"],
    prueba["probabilidad_churn"]
    * EXITO_CAMPANIA
    * prueba["margen_futuro_estimado"],
    0,
)
prueba["costo_campania"] = np.where(
    prueba["recomendar_contacto"],
    COSTO_CAMPANIA,
    0,
)
prueba["valor_neto_esperado"] = prueba["beneficio_esperado"] - prueba["costo_campania"]

cantidad_contactada = int(prueba["recomendar_contacto"].sum())
beneficio_total = prueba["beneficio_esperado"].sum()
costo_total = prueba["costo_campania"].sum()
valor_neto_total = prueba["valor_neto_esperado"].sum()

if costo_total > 0:
    roi_esperado = (valor_neto_total / costo_total) * 100
else:
    roi_esperado = 0

bajas_en_prueba = int(prueba["churn_real"].sum())
bajas_detectadas = int(
    ((prueba["churn_real"] == 1) & prueba["recomendar_contacto"]).sum()
)

if bajas_en_prueba > 0:
    recall_contactados = bajas_detectadas / bajas_en_prueba
else:
    recall_contactados = 0

print("Umbral elegido:", round(UMBRAL_ELEGIDO, 2))
print("Clientes de prueba que contactaría:", cantidad_contactada)
print("Beneficio esperado:", round(beneficio_total, 2))
print("Costo de campañas:", round(costo_total, 2))
print("Valor neto esperado:", round(valor_neto_total, 2))
print("ROI esperado (%):", round(roi_esperado, 2))
print("Recall de clientes que se fueron:", round(recall_contactados, 3))

In [ ]:
# Armé una tabla sencilla para revisar la decisión junto con el resultado real.
prueba["resultado_real"] = prueba["churn_real"].map({0: "No se fue", 1: "Se fue"})
matriz = pd.crosstab(
    prueba["resultado_real"],
    prueba["recomendar_contacto"],
    rownames=["Resultado real"],
    colnames=["Recomendé contactar"],
    margins=True,
)
display(matriz)

In [ ]:
# Guardé las decisiones del conjunto de prueba para poder revisarlas.
resultado_final = prueba[[
    "customer_id",
    "churn_real",
    "probabilidad_churn",
    "recomendar_contacto",
    "beneficio_esperado",
    "costo_campania",
    "valor_neto_esperado",
]].copy()

resultado_final.to_csv("decisiones_de_retencion.csv", index=False)
print("Guardé decisiones_de_retencion.csv")
display(resultado_final.head())

## Qué me quedó de esta parte

El umbral elegido fue el que dio el mayor valor neto esperado en validación. Luego lo revisé en prueba para tener una estimación separada. El valor neto y el ROI son simulados: para saber si una campaña realmente funciona necesitaría sus resultados observados.